# Heart risk results explorer

This notebook reads the artifacts produced by `scripts/train_models.py`. It does not retrain the models by default. Run the training script first if `reports/metrics.json` is missing.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
metrics_path = ROOT / "reports" / "metrics.json"
figures_dir = ROOT / "reports" / "figures"

if not metrics_path.exists():
    raise FileNotFoundError(
        "Run .venv\\Scripts\\python.exe scripts\\train_models.py first."
    )

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics.keys()

## Dataset and split

The outcome is self-reported coronary heart disease or myocardial infarction. The survey is cross-sectional, so these are risk associations rather than future-event predictions.

In [ ]:
pd.DataFrame([metrics["dataset"]])


In [ ]:
pd.DataFrame(
    [
        {"split": name, "n": values["n"], "prevalence": values["prevalence"]}
        for name, values in metrics["split"].items()
    ]
)


## Model comparison

The early model excludes diagnosed conditions and functional limitations. The full model adds them as an upper-bound benchmark.

In [ ]:
pd.DataFrame(metrics["model_comparison"])


## Test metrics for the selected models

In [ ]:
pd.DataFrame(
    {
        feature_set: metrics["models"][feature_set]["test"]
        for feature_set in ("early", "full")
    }
).T[
    [
        "threshold",
        "roc_auc",
        "pr_auc",
        "brier",
        "sensitivity",
        "specificity",
        "precision",
        "f2",
        "ece",
    ]
]


## Global SHAP importance

In [ ]:
for feature_set in ("early", "full"):
    print(feature_set.title())
    display(
        pd.DataFrame(metrics["models"][feature_set]["shap"]["importance"])
        .head(10)
        .style.format({"mean_abs_shap": "{:.3f}"})
    )


## Figures

In [ ]:
for name in (
    "model_comparison.png",
    "roc_pr_early.png",
    "shap_beeswarm_early.png",
    "shap_waterfall_early_high.png",
):
    path = figures_dir / name
    print(name)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing; run training.")


## Subgroup caveat

The early model is more sensitive for men than women in the held-out test set, and sensitivity is lower for some racial/ethnic groups. Review the subgroup tables alongside the confidence intervals and sample sizes before drawing conclusions.

In [ ]:
pd.DataFrame(metrics["models"]["early"]["subgroups"]["sex"])
